# 01 - Exploratory Data Analysis (Stage 3)
**Dataset:** `data/raw/online_shoppers_intention_synthetic_missing.xlsx` - Online Shoppers Purchasing Intention (UCI) with MCAR missing values in 4 columns.
**Goal:** understand structure, missing values, duplicates, outliers, class imbalance, relationships and leakage risk. Every observation here must be written down because it justifies a later preprocessing decision.

Run cells top to bottom. Figures are saved automatically to `reports/figures`, tables to `reports/tables`.

In [2]:
import os, sys, warnings
from pathlib import Path

# make `import src...` work no matter where Jupyter was started
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

# os.environ["FAST"] = "1"   # <- uncomment for a quick dry run (small sample, 3 folds). Must be BEFORE the src imports.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif

from src.preprocessing import (CATEGORICAL_COLS, LEAKAGE_SUSPECTS, MISSING_CATEGORICAL, MISSING_NUMERIC,
                               NUMERIC_COLS, TARGET, TABLE_DIR, basic_clean, load_modified)
from src.evaluate import save_fig, save_table

sns.set_theme(style="whitegrid")
raw = load_modified()
df = basic_clean(raw)
print("Shape:", df.shape)
df.head()

ImportError: cannot import name 'CATEGORICAL_COLS' from 'src.preprocessing' (C:\Users\Vindiya\Documents\GitHub\Online-shoppers-purchase-intent\src\preprocessing.py)

## 1. Structure and types

In [ ]:
structure = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_unique": df.nunique(),
                          "n_missing": df.isna().sum()}).reset_index(names="column")
save_table(structure, "eda_structure")
display(structure)
desc = df[NUMERIC_COLS].describe().T.reset_index(names="column")
save_table(desc, "eda_numeric_describe")
display(desc)

## 2. Missing values
Four columns contain missing values. We check how much, where, and whether the pattern looks random (MCAR).

In [ ]:
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss_tbl = pd.DataFrame({"missing": miss, "pct": (miss / len(df) * 100).round(2)}).reset_index(names="column")
save_table(miss_tbl, "eda_missing_counts")
display(miss_tbl)

fig, ax = plt.subplots(figsize=(7, 4))
(miss / len(df) * 100).plot.bar(ax=ax, color="tomato")
ax.set_ylabel("% missing"); ax.set_title("Missing values per column")
save_fig(fig, "eda_missing_bar")

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.sample(1500, random_state=1).sort_index().isna().T, cbar=False, ax=ax, yticklabels=True)
ax.set_title("Missingness matrix (1,500 random rows; dark = missing)")
save_fig(fig, "eda_missing_matrix")

In [ ]:
# MCAR check: is "missing" related to the target or to Month?  p > 0.05  => no evidence against MCAR
rows = []
for c in MISSING_NUMERIC + MISSING_CATEGORICAL:
    flag = df[c].isna()
    for other in [TARGET, "Month"]:
        p = chi2_contingency(pd.crosstab(flag, df[other]))[1]
        rows.append({"column": c, "compared_with": other, "chi2_p_value": round(p, 4),
                     "evidence_against_MCAR": p < 0.05})
mcar = pd.DataFrame(rows)
save_table(mcar, "eda_mcar_check")
mcar

## 3. Duplicates and class imbalance

In [ ]:
n_dup = int(df.duplicated().sum())
print(f"Duplicated rows: {n_dup} ({n_dup / len(df):.2%})")
save_table(pd.DataFrame({"duplicated_rows": [n_dup]}), "eda_duplicates")

vc = df[TARGET].value_counts()
print(vc.to_dict(), (vc / len(df)).round(3).to_dict())
fig, ax = plt.subplots(figsize=(4, 4))
sns.countplot(x=TARGET, data=df, ax=ax)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.0f}\n({p.get_height() / len(df):.1%})", (p.get_x() + .4, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Target balance (Revenue)"); ax.set_ylim(0, vc.max() * 1.15)
save_fig(fig, "eda_target_balance")

## 4. Distributions and outliers

In [ ]:
def grid(fn, title):
    fig, axes = plt.subplots(2, 5, figsize=(20, 7))
    for ax, c in zip(axes.ravel(), NUMERIC_COLS):
        fn(ax, c); ax.set_title(c)
    fig.suptitle(title); fig.tight_layout()
    return fig

save_fig(grid(lambda ax, c: sns.histplot(df[c].dropna(), bins=40, ax=ax), "Raw distributions"), "eda_hist_raw")
save_fig(grid(lambda ax, c: sns.histplot(np.log1p(df[c].dropna()), bins=40, ax=ax), "log1p distributions"), "eda_hist_log1p")
save_fig(grid(lambda ax, c: sns.boxplot(x=df[c], ax=ax), "Boxplots"), "eda_boxplots")

In [ ]:
out_rows = []
for c in NUMERIC_COLS:
    s = df[c].dropna()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    n_out = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
    out_rows.append({"column": c, "skew": round(s.skew(), 2), "zero_share_%": round((s == 0).mean() * 100, 1),
                     "IQR_outliers": n_out, "IQR_outlier_%": round(n_out / len(s) * 100, 1),
                     "p99": s.quantile(.99), "max": s.max()})
outliers = pd.DataFrame(out_rows)
save_table(outliers, "eda_outliers")
outliers

**Write down:** durations and counts are strongly right-skewed with many zeros, so the IQR rule flags many "outliers" that are real behaviour (long sessions), not errors. Plan: log1p transform or 1-99% winsorising rather than deleting rows (compared in notebook 02).

## 5. Relationships with the target

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for ax, c in zip(axes.ravel(), CATEGORICAL_COLS):
    rate = df.groupby(c)[TARGET].mean().sort_values(ascending=False)
    rate.plot.bar(ax=ax, color="steelblue")
    ax.axhline(df[TARGET].mean(), color="red", ls="--")
    ax.set_title(f"Purchase rate by {c}"); ax.set_ylabel("P(Revenue=1)")
axes.ravel()[-1].axis("off")
fig.tight_layout(); save_fig(fig, "eda_purchase_rate_by_category")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, c in zip(axes.ravel(), NUMERIC_COLS):
    sns.boxplot(x=TARGET, y=np.log1p(df[c]), data=df, ax=ax); ax.set_title(f"log1p({c}) vs Revenue")
fig.tight_layout(); save_fig(fig, "eda_numeric_vs_target")

In [ ]:
corr = df[NUMERIC_COLS + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax); ax.set_title("Correlation heatmap")
save_fig(fig, "eda_correlation_heatmap")
corr_t = (corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
          .rename("corr_with_revenue").rename_axis("feature").reset_index())
save_table(corr_t, "eda_corr_with_target")
corr_t

In [ ]:
# mutual information (quick relevance ranking; median-filled copy used ONLY here for EDA)
tmp = df.copy()
for c in NUMERIC_COLS:
    tmp[c] = tmp[c].fillna(tmp[c].median())
for c in CATEGORICAL_COLS:
    tmp[c] = tmp[c].fillna("Unknown").astype("category").cat.codes
cols = NUMERIC_COLS + CATEGORICAL_COLS
mi = pd.Series(mutual_info_classif(tmp[cols], tmp[TARGET], discrete_features=[False] * len(NUMERIC_COLS) + [True] * len(CATEGORICAL_COLS),
                                   random_state=42), index=cols).sort_values()
fig, ax = plt.subplots(figsize=(7, 6)); mi.plot.barh(ax=ax); ax.set_title("Mutual information with Revenue")
save_fig(fig, "eda_mutual_information")
save_table(mi.sort_values(ascending=False).rename("mutual_info").rename_axis("feature").reset_index(), "eda_mutual_info")

## 6. Data-leakage assessment

In [ ]:
notes = f"""# Data-leakage assessment
Feature leakage risk (target proxies / only known at session end):
- PageValues: average value of pages visited before a transaction; strongest predictor; computed by Google Analytics from
  e-commerce transactions so it partly encodes the outcome. Tested by dropping it in notebook 02.
- ExitRates / BounceRates: session-level aggregates, complete only when the session ends.
Procedural leakage and prevention:
- Imputers, scalers, encoders, capping quantiles, feature selection and SMOTE are fitted inside a Pipeline (training folds only).
- Stratified split BEFORE any fitting; duplicates removed before splitting.
- Missingness is unrelated to Revenue (chi-square, eda_mcar_check.csv).
Suspect columns: {LEAKAGE_SUSPECTS}
"""
(TABLE_DIR / "eda_leakage_notes.md").write_text(notes)
print(notes)

## 7. Findings checklist (fill in for your report and viva)
- Class balance: ~15.5% buyers -> accuracy misleading; use F1 / recall / PR-AUC.
- Missing: 4 columns, 4-7%, MCAR (p-values in table) -> imputation is valid, deletion wastes ~20% of rows.
- Skewed, zero-inflated numerics -> log1p / winsorise.
- Strongest relationships: PageValues, ExitRates, Month (Nov peak), VisitorType.
- Leakage: PageValues decision; pipeline-only fitting.